# 03 Gci Analysis

gCI / gAUC analysis notebook (Python)

Purpose: compute the proposed group-level discrimination metrics and compare them with pairwise fairness metrics.

# Initialize Truveta SDK

In [ ]:
from truveta.study import Client, OutputMode, display_df
import pyspark.pandas as ps
import pandas as pd
import numpy as np

In [ ]:
# Use only one statement below and comment out whichever you are not using.
client = Client(output_mode = OutputMode.PandasOnSpark)
# client = Client(output_mode = OutputMode.PySpark)

In [ ]:
study = client.get_study()
# Use only one statement below and comment out whichever you are not using.
# population = study.get_population(title = "PREVENT and PCE")
population = study.get_population(id = "<TRUVETA_POPULATION_ID>")
population

In [ ]:
# Get latest completed active snapshot.
snapshot = population.get_latest_snapshot()
snapshot

In [ ]:
# Show tables in the snapshot.
snapshot.get_tables()

# Prepare data

In [ ]:
df = study.load_artifacts_data("/PREVENT_PCE/Aim_1/original_prevent_result")
display_df(df)
df = df.to_pandas()

# Calculate gCI

In [ ]:
def gCI_spark(s_test, t_test, pred_risk,
              weights=None,
              pos_group=None,
              return_num_valid=False,
              tied_tol=1e-8):
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when, abs as abs_
    from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType

    # Initialize SparkSession
    spark = SparkSession.builder.getOrCreate()

    # Convert NumPy arrays to lists
    s_test_list = s_test.astype(int).tolist()
    t_test_list = t_test.astype(float).tolist()
    pred_risk_list = pred_risk.astype(float).tolist()
    
    if weights is not None:
        weights_list = weights.astype(float).tolist()
    else:
        weights_list = [1.0] * len(s_test_list)
    
    if pos_group is not None:
        pos_group_list = pos_group.astype(bool).tolist()
    else:
        pos_group_list = [True] * len(s_test_list)

    # Prepare data
    data = list(zip(s_test_list, t_test_list, pred_risk_list, weights_list, pos_group_list))
    data_with_id = [(i,) + row for i, row in enumerate(data)]

    schema = StructType([
        StructField('id', IntegerType(), False),
        StructField('s_test', IntegerType(), True),
        StructField('t_test', FloatType(), True),
        StructField('pred_risk', FloatType(), True),
        StructField('weights', FloatType(), True),
        StructField('pos_group', BooleanType(), True)
    ])

    df = spark.createDataFrame(data_with_id, schema=schema)

    # Positive group individuals
    df_pos = df.filter((col('pos_group') == True)).select(
        col('id').alias('id_pos'),
        col('s_test').alias('s_test_pos'),
        col('t_test').alias('t_test_pos'),
        col('pred_risk').alias('pred_risk_pos'),
        col('weights').alias('weights_pos')
    )

    # All individuals
    df_all = df.select(
        col('id').alias('id_all'),
        col('s_test').alias('s_test_all'),
        col('t_test').alias('t_test_all'),
        col('pred_risk').alias('pred_risk_all'),
        col('weights').alias('weights_all')
    )

    # Cross join
    df_pairs = df_pos.crossJoin(df_all)

    # Exclude self comparisons
    df_pairs = df_pairs.filter(col('id_pos') != col('id_all'))

    # Two ways to be comparable:
    # 1. pos earlier and event at pos
    cond1 = (col('t_test_pos') < col('t_test_all')) & (col('s_test_pos') == 1)
    # 2. all earlier and event at all
    cond2 = (col('t_test_all') < col('t_test_pos')) & (col('s_test_all') == 1)

    # Keep only comparable pairs
    df_valid = df_pairs.filter(cond1 | cond2)

    # Weight product
    df_valid = df_valid.withColumn('weight_product', col('weights_pos') * col('weights_all'))

    # Risk difference (always pred_pos - pred_all)
    df_valid = df_valid.withColumn('risk_diff', col('pred_risk_pos') - col('pred_risk_all'))

    # Determine concordance
    df_valid = df_valid.withColumn(
        'correctly_ranked',
        when((cond1 & (col('risk_diff') > tied_tol)) | (cond2 & (col('risk_diff') < -tied_tol)), 1.0).otherwise(0.0)
    ).withColumn(
        'tied',
        when(abs_(col('risk_diff')) <= tied_tol, 0.5).otherwise(0.0)
    )

    # Contribution
    df_valid = df_valid.withColumn(
        'contribution',
        (col('correctly_ranked') + col('tied')) * col('weight_product')
    )

    # Numerator and denominator
    numerator = df_valid.agg({'contribution': 'sum'}).collect()[0][0]
    denominator = df_valid.agg({'weight_product': 'sum'}).collect()[0][0]

    ci = numerator / denominator if denominator and denominator != 0 else None

    if return_num_valid:
        num_valid = df_valid.count()
        return ci, num_valid
    else:
        return ci


In [ ]:
def gCI_spark_fast(s_test, t_test, pred_risk,
                   weights=None,
                   pos_group=None,
                   return_num_valid=False,
                   tied_tol=1e-8):
    """
    Fast Spark implementation of group-specific Concordance Index (gCI) without full cross-join.
    Suitable for large-scale datasets (n > 100,000).
    """

    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when, abs as abs_
    from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType, IntegerType

    # Initialize SparkSession
    spark = SparkSession.builder.getOrCreate()

    # Convert to lists
    s_test_list = s_test.astype(int).tolist()
    t_test_list = t_test.astype(float).tolist()
    pred_risk_list = pred_risk.astype(float).tolist()
    
    if weights is not None:
        weights_list = weights.astype(float).tolist()
    else:
        weights_list = [1.0] * len(s_test_list)
    
    if pos_group is not None:
        pos_group_list = pos_group.astype(bool).tolist()
    else:
        pos_group_list = [True] * len(s_test_list)

    # Prepare DataFrame
    data = list(zip(s_test_list, t_test_list, pred_risk_list, weights_list, pos_group_list))
    data_with_id = [(i,) + row for i, row in enumerate(data)]

    schema = StructType([
        StructField('id', IntegerType(), False),
        StructField('s_test', IntegerType(), True),
        StructField('t_test', FloatType(), True),
        StructField('pred_risk', FloatType(), True),
        StructField('weights', FloatType(), True),
        StructField('pos_group', BooleanType(), True)
    ])

    df = spark.createDataFrame(data_with_id, schema=schema)

    # Group individuals (e.g., female)
    df_pos = df.filter(col('pos_group') == True).select(
        col('id').alias('id_pos'),
        col('s_test').alias('s_test_pos'),
        col('t_test').alias('t_test_pos'),
        col('pred_risk').alias('pred_risk_pos'),
        col('weights').alias('weights_pos')
    )

    # All individuals with event observed
    df_event = df.filter(col('s_test') == 1).select(
        col('id').alias('id_all'),
        col('s_test').alias('s_test_all'),
        col('t_test').alias('t_test_all'),
        col('pred_risk').alias('pred_risk_all'),
        col('weights').alias('weights_all')
    )

    ### Compare df_pos individuals earlier
    df_early = df_pos.alias('pos').join(
        df_event.alias('all'),
        on=(col('pos.t_test_pos') < col('all.t_test_all'))
    ).filter(
        col('pos.s_test_pos') == 1
    )

    df_early = df_early.withColumn('weight_product', col('weights_pos') * col('weights_all'))
    df_early = df_early.withColumn('risk_diff', col('pred_risk_pos') - col('pred_risk_all'))

    df_early = df_early.withColumn(
        'correctly_ranked',
        when(col('risk_diff') > tied_tol, 1.0).otherwise(0.0)
    ).withColumn(
        'tied',
        when(abs_(col('risk_diff')) <= tied_tol, 0.5).otherwise(0.0)
    ).withColumn(
        'contribution',
        (col('correctly_ranked') + col('tied')) * col('weight_product')
    )

    ### Compare df_event individuals earlier
    df_late = df_event.alias('all').join(
        df_pos.alias('pos'),
        on=(col('all.t_test_all') < col('pos.t_test_pos'))
    )

    df_late = df_late.withColumn('weight_product', col('weights_all') * col('weights_pos'))
    df_late = df_late.withColumn('risk_diff', col('pred_risk_pos') - col('pred_risk_all'))

    df_late = df_late.withColumn(
        'correctly_ranked',
        when(col('risk_diff') < -tied_tol, 1.0).otherwise(0.0)
    ).withColumn(
        'tied',
        when(abs_(col('risk_diff')) <= tied_tol, 0.5).otherwise(0.0)
    ).withColumn(
        'contribution',
        (col('correctly_ranked') + col('tied')) * col('weight_product')
    )

    # Union two parts
    df_valid = df_early.select('contribution', 'weight_product').union(
        df_late.select('contribution', 'weight_product')
    )

    # Aggregation
    result = df_valid.agg({'contribution': 'sum', 'weight_product': 'sum'}).collect()[0]
    numerator = result['sum(contribution)']
    denominator = result['sum(weight_product)']

    ci = numerator / denominator if denominator and denominator != 0 else None

    if return_num_valid:
        num_valid = df_valid.count()
        return ci, num_valid
    else:
        return ci


In [ ]:
s_ascvd = (df['ascvd'] == 1).astype(int).values
t_ascvd = df['fu_ascvd'].values
risk_ascvd = df['ascvd_predicted_probability_10y'].values
female = (df['female'] == 1).astype(int).values
male = (df['female'] == 0).astype(int).values


In [ ]:
gCI_spark_fast(s_ascvd, t_ascvd, risk_ascvd, pos_group=female)


In [ ]:
gCI_spark_fast(s_ascvd, t_ascvd, risk_ascvd, pos_group=male) 

In [ ]:
full_group =[
    "female", "ethnicity", "race", "cat_insurance", "cat_highest_edu_individual",
    "cat_med_nonadherence_risk", "cat_address_own_or_rent",
    "cat_current_address_dwelling_type", "cat_health_mgmt_nonmotivation_risk",
    "cat_highest_edu_household", "cat_healthcare_cost_risk", 
    "cat_address_change_60_months", 
    "cat_med_adherence_rate", "cat_current_address_median_income"
]
 
selected_groups_1 = [
    "female", "ethnicity", "race", "cat_insurance"]

selected_groups_2 = [
    "cat_highest_edu_individual",
    "cat_address_own_or_rent",
    "cat_current_address_dwelling_type", "cat_current_address_median_income"
]


s_ascvd = (df['ascvd'] == 1).astype(int).values
t_ascvd = df['fu_ascvd'].values
risk_ascvd = df['ascvd_predicted_probability_10y'].values

In [ ]:
gci_results = {}

# Loop over each group variable
for group_var in selected_groups_1:
    group_values = df[group_var].unique()  # get unique categories within group
    for val in group_values:
        # Create binary mask: 1 if equal to val, 0 otherwise
        pos_group = (df[group_var] == val).astype(int).values
        
        # Calculate gCI
        gci = gCI_spark_fast(
            s_ascvd,
            t_ascvd,
            risk_ascvd,
            pos_group=pos_group
        )
        
        # Save the result
        gci_results[(group_var, val)] = gci

# Display
for (group_var, val), gci in gci_results.items():
    print(f"gCI for {group_var} = {val}: {gci:.4f}")

# gAUC

In [ ]:
def gAUC_spark(y_test, pred_risk,
              weights=None,
              pos_group=None,
              return_num_valid=False,
              tied_tol=1e-8):
    """
    Group-conditioned AUC in the same spirit as your gCI:
      - anchor: CASES (y=1) inside pos_group
      - compare against: CONTROLS (y=0) from ALL individuals
      - metric: P( pred_case > pred_control ) with 0.5 for ties
      - weighting: product weights_case * weights_control

    Returns: gAUC (float) or (gAUC, num_valid_pairs) if return_num_valid=True
    """
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when, abs as abs_
    from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType

    spark = SparkSession.builder.getOrCreate()

    # --- to python lists (Spark DataFrame build) ---
    y_list   = y_test.astype(int).tolist()
    pr_list  = pred_risk.astype(float).tolist()

    n = len(y_list)
    if weights is not None:
        w_list = weights.astype(float).tolist()
    else:
        w_list = [1.0] * n

    if pos_group is not None:
        # accept {0,1}, bool, etc.
        pg_list = (pos_group.astype(bool)).tolist()
    else:
        pg_list = [True] * n

    data = list(zip(y_list, pr_list, w_list, pg_list))
    data_with_id = [(i,) + row for i, row in enumerate(data)]

    schema = StructType([
        StructField("id", IntegerType(), False),
        StructField("y", IntegerType(), True),
        StructField("pred_risk", FloatType(), True),
        StructField("weights", FloatType(), True),
        StructField("pos_group", BooleanType(), True),
    ])

    df = spark.createDataFrame(data_with_id, schema=schema)

    # --- anchor cases: in pos_group AND y==1 ---
    df_case = df.filter((col("pos_group") == True) & (col("y") == 1)).select(
        col("id").alias("id_case"),
        col("pred_risk").alias("pred_case"),
        col("weights").alias("w_case"),
    )

    # --- all controls: y==0 (from everyone) ---
    df_ctrl = df.filter(col("y") == 0).select(
        col("id").alias("id_ctrl"),
        col("pred_risk").alias("pred_ctrl"),
        col("weights").alias("w_ctrl"),
    )

    # Cross join (cases in group) x (all controls)
    df_pairs = df_case.crossJoin(df_ctrl)

    # Weight product
    df_pairs = df_pairs.withColumn("weight_product", col("w_case") * col("w_ctrl"))

    # Risk difference: case - control
    df_pairs = df_pairs.withColumn("risk_diff", col("pred_case") - col("pred_ctrl"))

    # Concordance for AUC
    df_pairs = df_pairs.withColumn(
        "correctly_ranked",
        when(col("risk_diff") > tied_tol, 1.0).otherwise(0.0)
    ).withColumn(
        "tied",
        when(abs_(col("risk_diff")) <= tied_tol, 0.5).otherwise(0.0)
    ).withColumn(
        "contribution",
        (col("correctly_ranked") + col("tied")) * col("weight_product")
    )

    # Numerator / Denominator
    numerator = df_pairs.agg({"contribution": "sum"}).first()[0]
    denominator = df_pairs.agg({"weight_product": "sum"}).first()[0]

    gauc = (numerator / denominator) if denominator and denominator != 0 else None

    if return_num_valid:
        num_valid = df_pairs.count()
        return gauc, num_valid
    else:
        return gauc


In [ ]:
gauc_results = {}

for group_var in selected_groups_2:
    group_values = pd.unique(df[group_var])

    for val in group_values:
        if pd.isna(val):
            continue

        pos_group = (df[group_var].values == val)

        gauc = gAUC_spark(
            y_test=s_ascvd,          # 0/1 array
            pred_risk=risk_ascvd,     # predicted probability/score
            pos_group=pos_group
        )

        gauc_results[(group_var, val)] = gauc

for (group_var, val), gauc in gauc_results.items():
    if gauc is None:
        print(f"gAUC for {group_var} = {val}: NA (no case-control pairs)")
    else:
        print(f"gAUC for {group_var} = {val}: {gauc:.4f}")


# xAUC

In [ ]:
def xAUC_spark(y_test, pred_risk,
              weights=None,
              pos_group=None, neg_group=None,
              return_num_valid=False,
              tied_tol=1e-8):
    """
    xAUC(pos, neg) = P( score(case in pos_group) > score(control in neg_group) )
      - cases:    y=1 AND pos_group==True
      - controls: y=0 AND neg_group==True
      - ties: 0.5
      - weights: product weights_case * weights_control
    """
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when, abs as abs_
    from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, BooleanType

    spark = SparkSession.builder.getOrCreate()

    # Convert inputs to python lists
    y_list  = y_test.astype(int).tolist()
    pr_list = pred_risk.astype(float).tolist()
    n = len(y_list)

    if weights is not None:
        w_list = weights.astype(float).tolist()
    else:
        w_list = [1.0] * n

    if pos_group is not None:
        pos_list = pos_group.astype(bool).tolist()
    else:
        pos_list = [True] * n

    if neg_group is not None:
        neg_list = neg_group.astype(bool).tolist()
    else:
        neg_list = [True] * n

    data = list(zip(y_list, pr_list, w_list, pos_list, neg_list))
    data_with_id = [(i,) + row for i, row in enumerate(data)]

    schema = StructType([
        StructField("id", IntegerType(), False),
        StructField("y", IntegerType(), True),
        StructField("pred_risk", FloatType(), True),
        StructField("weights", FloatType(), True),
        StructField("pos_group", BooleanType(), True),
        StructField("neg_group", BooleanType(), True),
    ])

    df = spark.createDataFrame(data_with_id, schema=schema)

    # cases in pos_group
    df_pos = (df
              .filter((col("y") == 1) & (col("pos_group") == True))
              .select(col("id").alias("id_pos"),
                      col("pred_risk").alias("pred_pos"),
                      col("weights").alias("w_pos")))

    # controls in neg_group
    df_neg = (df
              .filter((col("y") == 0) & (col("neg_group") == True))
              .select(col("id").alias("id_neg"),
                      col("pred_risk").alias("pred_neg"),
                      col("weights").alias("w_neg")))

    # If either side is empty, return None
    # (Spark doesn't have cheap emptiness checks; this is still usually fine)
    # You can remove these counts if you want pure-lazy execution.
    if df_pos.limit(1).count() == 0 or df_neg.limit(1).count() == 0:
        return (None, 0) if return_num_valid else None

    df_pairs = df_pos.crossJoin(df_neg)

    df_pairs = df_pairs.withColumn("weight_product", col("w_pos") * col("w_neg"))
    df_pairs = df_pairs.withColumn("risk_diff", col("pred_pos") - col("pred_neg"))

    df_pairs = (df_pairs
        .withColumn("correctly_ranked",
                    when(col("risk_diff") > tied_tol, 1.0).otherwise(0.0))
        .withColumn("tied",
                    when(abs_(col("risk_diff")) <= tied_tol, 0.5).otherwise(0.0))
        .withColumn("contribution",
                    (col("correctly_ranked") + col("tied")) * col("weight_product"))
    )

    numerator = df_pairs.agg({"contribution": "sum"}).first()[0]
    denominator = df_pairs.agg({"weight_product": "sum"}).first()[0]

    xauc = (numerator / denominator) if denominator and denominator != 0 else None

    if return_num_valid:
        num_valid = df_pairs.count()
        return xauc, num_valid
    else:
        return xauc


In [ ]:
xauc_results = {}

for group_var in selected_groups_2:
    levels = [v for v in pd.unique(df[group_var]) if not pd.isna(v)]
    levels = list(levels)

    for a in levels:          # pos_group level (cases)
        pos_mask = (df[group_var].values == a)

        for b in levels:      # neg_group level (controls)
            neg_mask = (df[group_var].values == b)

            xauc = xAUC_spark(
                y_test=s_ascvd,         # 0/1 numpy array aligned with df rows
                pred_risk=risk_ascvd,    # numpy array aligned with df rows
                weights=None,            # or weights_array if you have it
                pos_group=pos_mask,
                neg_group=neg_mask,
                tied_tol=1e-8
            )

            xauc_results[(group_var, a, b)] = xauc

# Display nicely
for (group_var, a, b), xauc in xauc_results.items():
    if xauc is None:
        print(f"xAUC for {group_var}: cases={a} vs controls={b}: NA")
    else:
        print(f"xAUC for {group_var}: cases={a} vs controls={b}: {xauc:.4f}")


# Calculate xCI for PCE

In [ ]:
pce_df = study.load_artifacts_data("/PREVENT_PCE/Aim_1/original_pce_result")
display_df(pce_df)
pce_df = pce_df.to_pandas()

In [ ]:
pce_s = (pce_df['ascvd'] == 1).astype(int).values
pce_t = pce_df['fu_ascvd'].values
pce_risk = pce_df['ascvd_predicted_probability_10y'].values

In [ ]:
for var in selected_groups_1:
    print(f"\n--- xCI for {var} ---")
    levels = pce_df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (pce_df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (pce_df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    pce_s, pce_t, pce_risk,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")


In [ ]:
for var in selected_groups_2:
    print(f"\n--- xCI for {var} ---")
    levels = pce_df[var].dropna().unique()
    levels = [l for l in sorted(levels) if l != "No Information"]  # exclude "No Information"

    for l1 in levels:
        g1 = (pce_df[var] == l1).astype(int).values
        for l2 in levels:
            g2 = (pce_df[var] == l2).astype(int).values

            try:
                xci_val = xCI_spark(
                    pce_s, pce_t, pce_risk,
                    pos_group=g1, neg_group=g2
                )
                print(f"xCI {var}: {l1} vs {l2} = {xci_val:.4f}")
            except Exception as e:
                print(f"xCI {var}: {l1} vs {l2} = ERROR ({e})")
